## 02 - Camada Silver

### Objetivo

Realizar a limpeza, padronização e validação dos dados
provenientes da camada Bronze.

### Origem

workspace.bronze.alunos_raw

### Destino

workspace.silver.alunos_tratados

In [0]:
from pyspark.sql import functions as F

In [0]:
df_silver = spark.table("workspace.bronze.alunos_raw")


In [0]:
#coluna de matrícula está como inteiro, converter
df_silver = df_silver.withColumn(
    "MATR_ALUNO",
    F.col("MATR_ALUNO").cast("string")
)

df_silver.printSchema() #verificando


In [0]:
#correção dos nomes dos municípios, que não estão padronizados
df_silver = df_silver.withColumn(
    "MUNICIPIO",
    F.trim(F.col("MUNICIPIO"))
)

In [0]:
#transformar maiúsculas e minúsculas
df_silver = df_silver.withColumn(
    "MUNICIPIO",
    F.initcap(F.col("MUNICIPIO"))
)
#aqui já fica com 151 municípios que antes eram 217


#verificando
display(df_silver.groupBy("MUNICIPIO").count().orderBy(F.desc("MUNICIPIO")))


In [0]:
#fazendo uma tabela para corrigir algumas cidades

dados_correcao = [
    ("Sao Mateus", "São Mateus"),
    ("SAO MATEUS", "São Mateus"),
    ("SÃO MATEUS", "São Mateus"),
    ("São Mateus", "São Mateus"),
    ("Conceicao Da Barra", "Conceição da Barra"),
    ("CONCEICAO DA BARRA", "Conceição da Barra"),
    ("Conceição da Barra", "Conceição da Barra"),
    ("Braço Do Rio (conceição Da Barra)", "Conceição da Barra"),
    ("Pedro Canario", "Pedro Canário"),
    ("PEDRO CANARIO", "Pedro Canário"),
    ("Nova Venecia", "Nova Venécia"),
    ("NOVA VENECIA", "Nova Venécia"),
    ("Vitoria", "Vitória"),
    ("VITORIA", "Vitória"),
    ("Posto Da Mata Municipio De Nova Vicosa", "Nova Vicosa")
]

df_correcao = spark.createDataFrame(
    dados_correcao,
    ["municipio_original", "municipio_padrao"]
)


In [0]:
df_silver = df_silver.join(
    df_correcao,
    df_silver["MUNICIPIO"] == df_correcao["municipio_original"],
    "left"
)

display(
    df_silver.select(
        "MUNICIPIO",
        "municipio_padrao"
    )
    .distinct()
)



In [0]:
df_silver = df_silver.withColumn(
    "MUNICIPIO",
    F.coalesce(
        F.col("municipio_padrao"),
        F.col("MUNICIPIO")
    )
)

df_silver = df_silver.drop("municipio_padrao")

df_silver.groupBy("MUNICIPIO") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

In [0]:
#municpipios inválidos que eu vi
municipios_invalidos = [
    "Rua",
    "Avenida",
    "Avenida Central",
    "Rua Marte",
    "Outros",
    "Municipal",
    "Espirito Santo"
]

df_silver = df_silver.withColumn(
    "STATUS_MUNICIPIO",
    F.when(
        F.col("MUNICIPIO").isin(municipios_invalidos),
        "INVALIDO"
    )
    .when(
        F.col("MUNICIPIO").isNull(),
        "AUSENTE"
    )
    .otherwise("VALIDO")
)

df_silver.groupBy("STATUS_MUNICIPIO").count().show()

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.alunos_tratados")

display(
    spark.table("workspace.silver.alunos_tratados")
)